# INFO361 – analyse av svar

Notebooken leser eksporten fra Supabase (én rad per deltaker, søknadene i kolonnen `soknader`)
og viser:

1. Oversikt over deltakerne
2. Hvor godt deltakerne traff fasiten
3. Hvem som ble kalt inn til intervju
4. Hvilke deler av søknadene som ble markert som KI
5. Alle svar, deltaker for deltaker

**Datafilen skal ikke inn i GitHub.** Legg CSV-filen i samme mappe som notebooken (eller endre
`DATA_PATH`). `*.csv` er utelatt fra git i `.gitignore`.

In [ ]:
import json
from html import escape
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

DATA_PATH = Path("svar.csv")  # eksport fra Supabase

# Fasit. Ligger bare her i analysen, aldri i nettsiden.
FASIT = {"S01": "menneske", "S02": "ki_redigert", "S03": "ki"}
SOKER = {"S01": "Benjamin", "S02": "Siri", "S03": "Ola"}
KATEGORI = {"menneske": "Menneske", "ki_redigert": "KI + redigert", "ki": "KI"}
KATEGORI_REKKEFOLGE = ["menneske", "ki_redigert", "ki"]

# Farger (fast rekkefølge, validert fargepalett): én farge per kategori
FARGE = {"menneske": "#2a78d6", "ki_redigert": "#1baf7a", "ki": "#eb6834"}
TEKST, TEKST_SVAK, RUTENETT = "#0b0b0b", "#52514e", "#e4e3df"

plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 10,
    "axes.edgecolor": RUTENETT,
    "axes.labelcolor": TEKST_SVAK,
    "xtick.color": TEKST_SVAK,
    "ytick.color": TEKST_SVAK,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlelocation": "left",
    "axes.titleweight": "bold",
    "axes.titlecolor": TEKST,
})

def soknad_navn(app_id):
    return f"{app_id} {SOKER[app_id]} ({KATEGORI[FASIT[app_id]]})"

## Søknadstekstene

Samme tekster som i `src/data/applications.js`. De trengs for å vise markeringene i teksten.
Hvis tekstene på nettsiden endres, må de oppdateres her også.

In [ ]:
TEKSTER = {
    "S01": """Hei,

Jeg heter Benjamin, er 25 år gammel og kommer fra Bergen. Etter å ha lest meg opp, ser jeg at dere har et veldig engasjerende miljø med gode muligheter til å utvikle seg videre. Jeg er ivrig og har mye erfaring med teamarbeid, og tror derfor jeg kunne passet godt inn i denne stillingen.

Jeg har en bachelor innen administrasjon og ledelse fra Høgskolen på Vestlandet. Dette startet jeg fordi jeg trives godt å jobbe med mennesker og liker å jobbe systematisk. Tidligere har jeg vært avdelingsleder på Sabrura ved siden av studiene. Denne erfaringen har gitt meg god kompetanse innen samarbeid og ledelse. Her lærte jeg å løse mindre og større oppgaver, både for ansatte og besøkende i restauranten.

Jeg er utadvendt, aktiv og liker å være engasjert i sosiale miljøer rundt meg. Dette er noen av grunnene til at jeg meldte meg som leder i fadderstyret. Her fikk jeg nyttig erfaring med å organisere faddergrupper, opprettholde kontakt med bedrifter og ha overordnet blikk på flere tråder samtidig.

Med vennlig hilsen,
Benjamin Olsen""",
    "S02": """Hei,

Jeg heter Siri, er 24 år, fra Molde og har en bachelor i administrasjon og ledelse fra Høgskolen på Vestlandet. Stillingen som prosjektkoordinator hos Viro virker spennende fordi jeg ønsker å bruke utdanningen min i en praktisk rolle med ansvar for planlegging og oppfølging. Muligheten til å bidra i ulike prosjekter, samarbeide med andre og utvikle meg faglig gjør at denne rollen hadde kledd meg godt.

Som bemanningsansvarlig for studentbaren Kronbar, fikk jeg erfaring med koordinering og samarbeid i et aktivt studentmiljø. Rollen krevde oversikt, tydelig kommunikasjon og evne til å håndtere ulike situasjoner. Erfaringen har gitt meg et godt grunnlag for oppfølging av arbeidsoppgaver og at aktiviteter gjennomføres som planlagt.

Jeg er strukturert og trives med å arbeide både selvstendig og i team. Gjennom engasjementet i Kronbar har jeg fått erfaring med å møte ulike mennesker og bidra til et godt sosialt miljø. Jeg tar gjerne initiativ, er opptatt av å finne gode løsninger og ønsker å være en kollega andre kan stole på. Hos Viro vil jeg bidra med engasjement, godt samarbeid og vilje til å lære.

Med vennlig hilsen,
Siri Hansen""",
    "S03": """Hei,

Jeg heter Ola, er 26 år og kommer fra Oslo. Med en bachelor i administrasjon og ledelse fra Høgskulen på Vestlandet ønsker jeg nå å bruke utdanningen min i en praktisk og variert arbeidshverdag. Stillingen som prosjektkoordinator hos Viro virker spesielt interessant fordi den kombinerer struktur, samarbeid og koordinering – områder jeg trives godt med og ønsker å utvikle meg videre innen.

Gjennom erfaring som redaktør for studentradioen har jeg fått god erfaring med å koordinere mennesker, planlegge aktiviteter og følge opp oppgaver og tidsfrister. Rollen krevde tydelig kommunikasjon og evnen til å holde oversikt over flere oppgaver samtidig. Som skiftleder hos Digg Pizza har jeg i tillegg fått erfaring med ansvar, problemløsning og koordinering i en hektisk arbeidshverdag.

Jeg er strukturert, initiativrik og liker å samarbeide med andre. Gjennom både studentradioen og arbeidslivet har jeg blitt vant til å møte ulike mennesker og bidra til et godt arbeidsmiljø. Jeg arbeider samtidig godt selvstendig og tar gjerne ansvar for å få oppgaver gjennomført.

Med vennlig hilsen,
Ola Berg""",
}

## Les inn data

Lager tre tabeller:

- `deltakere`: én rad per deltaker
- `svar`: én rad per deltaker og søknad
- `markeringer`: én rad per markert tekstbit

In [ ]:
raw = pd.read_csv(DATA_PATH)

deltakere = raw[["participant_id", "gender", "age_group", "submitted_at"]].copy()
deltakere["submitted_at"] = pd.to_datetime(deltakere["submitted_at"], format="ISO8601")
deltakere = deltakere.sort_values("submitted_at").reset_index(drop=True)
deltakere["deltaker"] = [f"D{i + 1}" for i in range(len(deltakere))]  # kort navn til visning
kortnavn = dict(zip(deltakere["participant_id"], deltakere["deltaker"]))

svar_rader, markering_rader = [], []
for _, rad in raw.iterrows():
    for s in json.loads(rad["soknader"]):
        svar_rader.append({
            "participant_id": rad["participant_id"],
            "deltaker": kortnavn[rad["participant_id"]],
            "application_id": s["application_id"],
            "display_order": s["display_order"],
            "judgement": s["judgement"],
            "invited_to_interview": s["invited_to_interview"],
            "reasoning": s["reasoning"],
            "antall_markeringer": len(s["highlights"]),
        })
        for h in s["highlights"]:
            markering_rader.append({
                "participant_id": rad["participant_id"],
                "deltaker": kortnavn[rad["participant_id"]],
                "application_id": s["application_id"],
                "judgement": s["judgement"],
                **h,
            })

svar = pd.DataFrame(svar_rader)
svar["fasit"] = svar["application_id"].map(FASIT)
svar["riktig"] = svar["judgement"] == svar["fasit"]
# Mildere mål: så deltakeren at KI var involvert (ki/ki_redigert) eller ikke (menneske)?
svar["ki_involvering_riktig"] = (svar["judgement"] != "menneske") == (svar["fasit"] != "menneske")

markeringer = pd.DataFrame(markering_rader)

# Kontroll: stemmer posisjonene med søknadsteksten?
feil = [
    (m.deltaker, m.application_id, m.start)
    for m in markeringer.itertuples()
    if TEKSTER[m.application_id][m.start:m.end] != m.text
]
print(f"{len(deltakere)} deltakere, {len(svar)} svar, {len(markeringer)} markeringer")
print("Alle markeringer stemmer med teksten." if not feil else f"Avvik i markeringer: {feil}")

## 1. Deltakerne

In [ ]:
def fordeling(kolonne, navn):
    t = deltakere[kolonne].value_counts().rename_axis(navn).reset_index(name="Antall")
    return t.sort_values(navn).reset_index(drop=True)

display(HTML("<div style='display:flex;gap:40px;flex-wrap:wrap'>"
             + fordeling("gender", "Kjønn").to_html(index=False)
             + fordeling("age_group", "Aldersgruppe").to_html(index=False)
             + "</div>"))
print(f"Innsendt mellom {deltakere.submitted_at.min():%d.%m.%Y} og {deltakere.submitted_at.max():%d.%m.%Y}")

## 2. Traff deltakerne fasiten?

**Riktig** betyr at deltakeren valgte nøyaktig riktig kategori.
**KI-involvering riktig** er et mildere mål: deltakeren så om KI var involvert eller ikke,
uten å skille mellom «KI» og «KI + redigert».

In [ ]:
treff = (
    svar.groupby("application_id")
    .agg(n=("riktig", "size"), riktig=("riktig", "sum"), ki_involvering=("ki_involvering_riktig", "sum"))
    .reset_index()
)
treff["Søknad"] = treff["application_id"].map(soknad_navn)
treff["Riktig kategori"] = treff.apply(lambda r: f"{r.riktig} av {r.n} ({r.riktig / r.n:.0%})", axis=1)
treff["KI-involvering riktig"] = treff.apply(lambda r: f"{r.ki_involvering} av {r.n} ({r.ki_involvering / r.n:.0%})", axis=1)
display(treff[["Søknad", "Riktig kategori", "KI-involvering riktig"]].style.hide(axis="index"))

totalt = svar["riktig"].mean()
print(f"Totalt riktig kategori: {svar['riktig'].sum()} av {len(svar)} ({totalt:.0%}). "
      f"Ved ren gjetning er forventet treff 33 %.")

### Fasit mot deltakernes svar

In [ ]:
tabell = pd.crosstab(
    svar["fasit"], svar["judgement"]
).reindex(index=KATEGORI_REKKEFOLGE, columns=KATEGORI_REKKEFOLGE, fill_value=0)

fig, ax = plt.subplots(figsize=(5.6, 3.6))
ax.imshow(tabell.values, cmap="Blues", vmin=0, vmax=max(tabell.values.max(), 1))
for (i, j), v in np.ndenumerate(tabell.values):
    lys = v > tabell.values.max() * 0.55
    ax.text(j, i, str(v), ha="center", va="center", fontsize=13,
            color="white" if lys else TEKST, fontweight="bold" if i == j else "normal")
ax.set_xticks(range(3), [KATEGORI[k] for k in KATEGORI_REKKEFOLGE])
ax.set_yticks(range(3), [f"{KATEGORI[k]}\n({SOKER[a]})" for k in KATEGORI_REKKEFOLGE
                          for a in FASIT if FASIT[a] == k])
ax.set_xlabel("Deltakerens svar")
ax.set_ylabel("Fasit")
ax.set_title("Antall svar: fasit mot deltakerens svar")
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
ax.set_xticks(np.arange(-.5, 3), minor=True)
ax.set_yticks(np.arange(-.5, 3), minor=True)
ax.grid(which="minor", color="white", linewidth=2)
ax.tick_params(which="minor", length=0)
plt.tight_layout()
plt.show()
print("Diagonalen (uthevet) er riktige svar.")

### Hva svarte deltakerne på hver søknad?

In [ ]:
fordelt = (
    pd.crosstab(svar["application_id"], svar["judgement"])
    .reindex(columns=KATEGORI_REKKEFOLGE, fill_value=0)
    .sort_index(ascending=False)
)

fig, ax = plt.subplots(figsize=(7.5, 2.8))
venstre = np.zeros(len(fordelt))
for kat in KATEGORI_REKKEFOLGE:
    verdier = fordelt[kat].values
    ax.barh(range(len(fordelt)), verdier, left=venstre, color=FARGE[kat],
            edgecolor="white", linewidth=2, height=0.6, label=KATEGORI[kat])
    for i, v in enumerate(verdier):
        if v:
            ax.text(venstre[i] + v / 2, i, str(v), ha="center", va="center", color="white", fontweight="bold")
    venstre += verdier
ax.set_yticks(range(len(fordelt)), [soknad_navn(a) for a in fordelt.index])
ax.set_xlabel("Antall deltakere")
ax.xaxis.get_major_locator().set_params(integer=True)
ax.set_title("Deltakernes svar per søknad")
ax.legend(title="Deltakerens svar", frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.tick_params(axis="y", length=0)
plt.tight_layout()
plt.show()

### Antall riktige per deltaker

In [ ]:
per_deltaker = (
    svar.groupby("deltaker")
    .agg(riktig=("riktig", "sum"), ki_involvering=("ki_involvering_riktig", "sum"), markeringer=("antall_markeringer", "sum"))
    .reindex(deltakere["deltaker"])
    .reset_index()
    .merge(deltakere[["deltaker", "gender", "age_group"]], on="deltaker")
)
per_deltaker.columns = ["Deltaker", "Riktig kategori (av 3)", "KI-involvering riktig (av 3)", "Markeringer", "Kjønn", "Aldersgruppe"]
display(per_deltaker.style.hide(axis="index"))

### Påvirker rekkefølgen treffsikkerheten?

In [ ]:
rekkefolge = svar.groupby("display_order")["riktig"].agg(["sum", "size"]).reset_index()
rekkefolge["Riktig"] = rekkefolge.apply(lambda r: f"{r['sum']} av {r['size']} ({r['sum'] / r['size']:.0%})", axis=1)
rekkefolge = rekkefolge.rename(columns={"display_order": "Vist som nr."})[["Vist som nr.", "Riktig"]]
display(rekkefolge.style.hide(axis="index"))

## 3. Hvem ble kalt inn til intervju?

In [ ]:
intervju = svar.groupby("application_id")["invited_to_interview"].sum().reindex(FASIT.keys())
n = len(deltakere)

fig, ax = plt.subplots(figsize=(6.5, 2.4))
y = range(len(intervju))[::-1]
ax.barh(list(y), intervju.values, color=[FARGE[FASIT[a]] for a in intervju.index], height=0.55)
for yi, v in zip(y, intervju.values):
    ax.text(v + 0.05, yi, f"{v} av {n}", va="center", color=TEKST)
ax.set_yticks(list(y), [soknad_navn(a) for a in intervju.index])
ax.set_xlim(0, n + 0.8)
ax.set_xlabel("Antall deltakere som ville kalt inn søkeren")
ax.xaxis.get_major_locator().set_params(integer=True)
ax.set_title("Kalt inn til intervju")
ax.tick_params(axis="y", length=0)
plt.tight_layout()
plt.show()

ingen = (svar.groupby("deltaker")["invited_to_interview"].sum() == 0).sum()
print(f"{ingen} deltaker(e) ville ikke kalt inn noen.")

kryss = pd.crosstab(svar["judgement"].map(KATEGORI), svar["invited_to_interview"].map({True: "Kalt inn", False: "Ikke kalt inn"}))
kryss.index.name = "Deltakeren trodde søknaden var"
kryss.columns.name = None
display(kryss)

## 4. Hva ble markert som KI?

Fargen viser hvor mange deltakere som markerte hvert tegn. Jo mørkere, jo flere.
Hold musepekeren over en markert del for å se antallet.

In [ ]:
def varmekart(app_id):
    tekst = TEKSTER[app_id]
    teller = np.zeros(len(tekst), dtype=int)
    for m in markeringer[markeringer.application_id == app_id].itertuples():
        teller[m.start:m.end] += 1
    maks = max(len(deltakere), 1)

    deler, i = [], 0
    while i < len(tekst):
        j = i
        while j < len(tekst) and teller[j] == teller[i]:
            j += 1
        bit = escape(tekst[i:j])
        if teller[i]:
            alfa = 0.15 + 0.75 * teller[i] / maks
            farge = "white" if alfa > 0.6 else "#0b0b0b"
            bit = (f"<span title='{teller[i]} av {maks} markerte' "
                   f"style='background:rgba(42,120,214,{alfa:.2f});color:{farge};border-radius:2px'>{bit}</span>")
        deler.append(bit.replace("\n", "<br>"))
        i = j

    antall = (markeringer.application_id == app_id).sum()
    return (f"<h4 style='margin:18px 0 6px'>{escape(soknad_navn(app_id))} – {antall} markeringer</h4>"
            f"<div style='font-family:Georgia,serif;line-height:1.6;max-width:760px;"
            f"padding:14px 16px;border:1px solid #e4e3df;border-radius:8px;background:#fcfcfb;color:#0b0b0b'>"
            f"{''.join(deler)}</div>")

forklaring = "".join(
    f"<span style='display:inline-block;padding:2px 8px;margin-right:4px;border-radius:2px;"
    f"background:rgba(42,120,214,{0.15 + 0.75 * k / max(len(deltakere), 1):.2f});"
    f"color:{'white' if 0.15 + 0.75 * k / max(len(deltakere), 1) > 0.6 else '#0b0b0b'}'>{k}</span>"
    for k in range(1, len(deltakere) + 1)
)
display(HTML(f"<p>Antall deltakere som markerte: {forklaring}</p>"
             + "".join(varmekart(a) for a in FASIT)))

### Alle markeringer med kommentar

In [ ]:
pd.set_option("display.max_colwidth", None)
for app_id in FASIT:
    utvalg = markeringer[markeringer.application_id == app_id].sort_values("start")
    if utvalg.empty:
        continue
    display(HTML(f"<h4>{escape(soknad_navn(app_id))}</h4>"))
    display(
        utvalg[["deltaker", "text", "comment"]]
        .rename(columns={"deltaker": "Deltaker", "text": "Markert tekst", "comment": "Kommentar"})
        .style.hide(axis="index")
        .set_properties(**{"text-align": "left", "vertical-align": "top"})
    )

### Begrunnelser per søknad

In [ ]:
for app_id in FASIT:
    utvalg = svar[svar.application_id == app_id].copy()
    utvalg["Svar"] = utvalg["judgement"].map(KATEGORI)
    utvalg["Riktig"] = utvalg["riktig"].map({True: "✓ Riktig", False: "✗ Feil"})
    display(HTML(f"<h4>{escape(soknad_navn(app_id))}</h4>"))
    display(
        utvalg[["deltaker", "Svar", "Riktig", "reasoning"]]
        .rename(columns={"deltaker": "Deltaker", "reasoning": "Begrunnelse"})
        .style.hide(axis="index")
        .set_properties(**{"text-align": "left", "vertical-align": "top"})
    )

## 5. Alle svar, deltaker for deltaker

In [ ]:
def kort(rad):
    d = rad
    deler = [
        "<div style='border:1px solid #e4e3df;border-radius:10px;padding:14px 18px;margin:14px 0;"
        "max-width:820px;background:#fcfcfb;color:#0b0b0b'>",
        f"<div style='font-size:1.1em;font-weight:bold'>{d.deltaker}</div>",
        f"<div style='color:#52514e;font-size:.9em;margin-bottom:8px'>{escape(d.gender)} · {escape(d.age_group)} · "
        f"innsendt {d.submitted_at:%d.%m.%Y %H:%M} UTC · <code>{d.participant_id}</code></div>",
    ]
    for s in svar[svar.participant_id == d.participant_id].sort_values("display_order").itertuples():
        merke = "✓ Riktig" if s.riktig else "✗ Feil"
        merke_farge = "#006300" if s.riktig else "#b3261e"
        intervju = " · kalt inn til intervju" if s.invited_to_interview else ""
        deler.append(
            f"<div style='border-left:4px solid {FARGE[s.fasit]};padding:6px 12px;margin:10px 0'>"
            f"<div><b>{s.display_order}. {escape(soknad_navn(s.application_id))}</b>{intervju}</div>"
            f"<div>Svar: <b>{KATEGORI[s.judgement]}</b> "
            f"<span style='color:{merke_farge};font-weight:bold'>{merke}</span></div>"
            f"<div style='margin-top:4px;white-space:pre-wrap'><i>Begrunnelse:</i> {escape(s.reasoning) or '–'}</div>"
        )
        hs = markeringer[(markeringer.participant_id == d.participant_id) & (markeringer.application_id == s.application_id)]
        if len(hs):
            deler.append("<ul style='margin:6px 0 0 0'>")
            for h in hs.sort_values("start").itertuples():
                kommentar = f" — {escape(h.comment)}" if h.comment else ""
                deler.append(f"<li><mark style='background:#cfe2f8'>«{escape(h.text)}»</mark>{kommentar}</li>")
            deler.append("</ul>")
        deler.append("</div>")
    deler.append("</div>")
    return "".join(deler)

display(HTML("".join(kort(d) for d in deltakere.itertuples())))